# Week 10｜從計數到向量：關鍵字檢索 vs 語意檢索
**From counts to dense vectors: keyword retrieval vs semantic retrieval**

今天要回答一個問題：**使用者問了一個校規問題，電腦要如何從我們第 8 週切好的 chunks 中，找出最相關的那幾段？**

這一步驟名為「檢索 (retrieval)」，是期末校規問答系統 (RAG) 的前半段：先找到正確的條文區塊，後面才能請語言模型根據條文回答。

| 節次 | 內容 | 任務 |
|---|---|---|
| 第 1 節  | **先照 `10_setup.ipynb` 的 課前設定 (setup) 部分安裝套件、下載模型**；關鍵字檢索：jieba 斷詞、TF-IDF、BM25 | 1-A 教師示範、1-B  |
| 第 2 節 | 語意檢索：詞向量觀念、句向量、cosine 相似度 | 2-A 、2-B  |
| 第 3 節 | 用 10 個校規問題比較兩種檢索，繳交 | 3-A 、3-B 、3-C  |

**今天要繳交 (下課前 push)：**
1. `10_vector_similarity.ipynb`（保留輸出）
2. `week10_retrieval_comparison.json`（最後一格自動產生）
3. `AI_USAGE.md` 的 Week 10 段落
4. 更新後的 `requirements.txt` 和 `.gitignore`（由 `10_setup.ipynb` 自動完成）
5. 更新後的 `README.md` ：新增 **Week 10** 的說明（本週完成的內容、新增的檔案）

標有 `# TODO` 的地方是你要完成的程式碼 (7 個)。每個 TODO 後面都有一格 `assert` 檢查，執行後沒有錯誤訊息才算完成。

## 0. 開始之前

**上課第一件事：打開 `10_setup.ipynb`，照著課前設定 (setup) 部分完成安裝套件、下載模型 。**

1. 在 VS Code 終端機（確認有 `(.venv)`）執行兩行 `pip install`。
2. 從自主學習雲端學院下載句向量模型，解壓縮到專案的 `models/` 資料夾。

`pip install` 完成後，就可以繼續執行這份 notebook；第二行讓它在終端機背景執行。`10_setup.ipynb` 的 其餘部分（Run All）在第 2 節開始前完成即可。

### 檢查套件、設定專案路徑、載入 chunks

下一格檢查第 1 節需要的套件是否已經安裝。

In [107]:
import sys
import importlib.util

print("這份 notebook 使用的 Python：", sys.executable)

packages = {"jieba": "jieba", "rank_bm25": "rank_bm25", "sklearn": "scikit-learn", "pandas": "pandas"}
not_found = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]

assert not not_found, (
    f"找不到這些套件：{not_found}。請在 VS Code 終端機（確認有 (.venv)）執行："
    "pip install jieba rank_bm25 scikit-learn pandas，並確認右上角的 kernel 選的是 .venv。"
)
print("第 1 節需要的套件都已安裝")

這份 notebook 使用的 Python： c:\Users\user\Desktop\weiwei\U12627006-mcut\.venv\Scripts\python.exe
第 1 節需要的套件都已安裝


In [108]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import os

# 檢查目前工作目錄
current_dir = os.getcwd()
print(f"目前工作目錄: {current_dir}")

# 取得目前 Notebook 的絕對路徑
notebook_dir = os.path.abspath('')

# 往上推兩層取得根目錄的絕對路徑
# project_root = os.path.dirname(os.path.dirname(notebook_dir))

# 往上推一層取得根目錄的絕對路徑
project_root = os.path.dirname(notebook_dir)

# 組合出 data/processed 的絕對路徑
PROCESSED_DATA_DIR = Path(os.path.join(project_root, "data", "processed"))

ROOT = Path(project_root)
print(f"專案根目錄: {ROOT}")
assert PROCESSED_DATA_DIR.exists(), "找不到 data/processed。請檢查上面 project_root 是往上推兩層還是一層，並確認第 8 週的 chunks 已經產生。"
print("找到 data/processed")

目前工作目錄: c:\Users\user\Desktop\weiwei\U12627006-mcut\notebooks
專案根目錄: c:\Users\user\Desktop\weiwei\U12627006-mcut
找到 data/processed


In [109]:
CLUSTERS = ["Academic_Affairs", "Library_Information", "Student_Affairs"]

rows = []
for cluster in CLUSTERS:
    for path in sorted((PROCESSED_DATA_DIR / cluster).glob("cleaned_*_processed.json")):
        chunks = json.loads(path.read_text(encoding="utf-8"))
        doc = path.stem.removeprefix("cleaned_").removesuffix("_processed")
        for i, c in enumerate(chunks):
            meta = c.get("metadata", {})
            rows.append({
                "chunk_id": f"{doc}#{i}",
                "cluster": meta.get("cluster", cluster),
                "source_file": meta.get("source_file", path.name),
                "heading": " / ".join(meta.get("headings") or []),
                "text": c["text"],
            })

df = pd.DataFrame(rows)
assert len(df) > 0, "沒有讀到任何 chunk，請檢查 data/processed/<類別>/cleaned_*_processed.json 是否存在。"
texts = df["text"].tolist()

print(f"共載入 {len(df)} 個 chunks，來自 {df['source_file'].nunique()} 份文件")
df.groupby("cluster").size().rename("chunks").to_frame()

共載入 345 個 chunks，來自 9 份文件


,chunks
cluster,
Academic_Affairs,65
Library_Information,225
Student_Affairs,55


In [110]:
# 看其中一個 chunk 長什麼樣子
print(df.iloc[2]["chunk_id"])
print(df.iloc[2]["text"])

A231060004_學生申請期中停修課程實施要點#2
## 第二條 適用範圍

每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修後至少仍應修習一個科目。


In [111]:
# chunks 的長度差很多：有些只有一句話，有些是好幾頁的條文或表格
df["n_chars"] = df["text"].str.len()
print(df["n_chars"].describe().round(0).to_string())
print("超過 1000 字的 chunks：", int((df["n_chars"] > 1000).sum()), "個")

count      345.0
mean       953.0
std       1868.0
min          8.0
25%        119.0
50%        272.0
75%        859.0
max      14350.0
超過 1000 字的 chunks： 79 個


In [112]:
# 今天會一直用到的顯示函式：把檢索結果印出來
# top 的格式是 [(chunk 的列號, 分數), ...]，分數由高到低
def show(query, top, k=3):
    print(f"問題：{query}")
    for rank, (i, score) in enumerate(top[:k], 1):
        r = df.iloc[i]
        preview = r["text"].replace("\n", " ")[:70]
        print(f"  {rank}. [{score:.3f}] {r['cluster']}｜{r['source_file']}")
        print(f"       {preview}…")

---
# 第 1 節 Session 1｜關鍵字檢索 

## 1-1｜今天進行大學規章辦法問答系統 (RAG) 哪一部分的開發？

大學規章辦法問答系統 (RAG) 分成兩個階段。今天和下週(11週)進行**第一階段：檢索**的實作。

```
使用者的問題
     │
     ▼
【檢索 Retrieval】 從 346 個 chunks 中找出最相關的 3 個   ← 第 10、11 週
     │
     ▼
【生成 Generation】把問題和這 3 個 chunks 一起交給語言模型，請它根據條文回答   ← 第 12 週
     │
     ▼
答案（附上引用的條文）
```

**為什麼不把全部校規直接交給語言模型？** 因為校規太長，語言模型一次能讀的字數有限，而且讀得越多、越貴、越容易答錯。所以要先「找對條文」。**如果檢索找錯了條文，後面的語言模型再強，也只能根據錯的條文回答。**



## 1-2｜電腦如何比較兩段文字？

電腦不懂文字，只會算數字。所以第一步永遠是：**把文字變成一串數字（向量）**，再比較兩串數字有多接近。

最直接的做法是「數詞」，叫做**詞袋 (Bag of Words)**：每個詞是一個欄位，這個詞在文件中出現幾次，就填幾。下面用三份很短的「文件」示範（詞已經先用空白隔開）。

In [113]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

toy_docs = ["學生 申請 停修 科目", "學生 申請 請假 請假", "學生 圖書館 借書 期限"]
names = ["文件A", "文件B", "文件C"]

cv = CountVectorizer(token_pattern=r"\S+")
pd.DataFrame(cv.fit_transform(toy_docs).toarray(), index=names, columns=cv.get_feature_names_out())

,借書,停修,圖書館,學生,期限,申請,科目,請假
文件A,0,1,0,1,0,1,1,0
文件B,0,0,0,1,0,1,0,2
文件C,1,0,1,1,1,0,0,0


**如何讀這張表：** 每一列是一份文件，每一欄是一個詞。文件 B 的「請假」出現 2 次，所以填 2。每一列就是那份文件的**向量**。

**詞袋的問題：** 「學生」在三份文件中都出現。它對「分辨這三份文件」完全沒有幫助，但在詞袋中，它和「停修」一樣重要（都是 1）。

## 1-3｜TF-IDF：稀有的詞比較重要

**TF-IDF = TF × IDF**

- **TF (Term Frequency，詞頻)**：這個詞在**這份文件**中出現幾次。出現越多次，越重要。
- **IDF (Inverse Document Frequency，逆文件頻率)**：這個詞在**多少份文件**中出現過。出現在越多文件中，越不稀有，權重越低。

$$\text{IDF}(詞) = \log\frac{文件總數}{含有這個詞的文件數}$$

（scikit-learn 實際使用的公式多了平滑項，觀念相同。）

In [114]:
tv = TfidfVectorizer(token_pattern=r"\S+")
pd.DataFrame(tv.fit_transform(toy_docs).toarray().round(2), index=names, columns=tv.get_feature_names_out())

,借書,停修,圖書館,學生,期限,申請,科目,請假
文件A,0.00,0.58,0.00,0.35,0.00,0.44,0.58,0.0
文件B,0.00,0.00,0.00,0.27,0.00,0.34,0.00,0.9
文件C,0.55,0.00,0.55,0.32,0.55,0.00,0.00,0.0


**觀察文件 A 這一列：** 四個詞在文件 A 中都只出現 1 次，但是權重不同。

| 詞 | 出現在幾份文件 | 文件 A 中的權重 |
|---|---|---|
| 學生 | 3 份（全部） | 最低 |
| 申請 | 2 份 | 中間 |
| 停修、科目 | 1 份（只有文件 A） | 最高 |

這就是 TF-IDF 的想法：**越能代表這份文件的詞，權重越高。**



## 1-4｜BM25：TF-IDF 的改良版

BM25 是搜尋引擎長年使用的標準方法。它和 TF-IDF 一樣，一個詞的分數是「**IDF × TF 部分**」。IDF 的想法相同（越稀有的詞越重要），**BM25 改的是 TF 部分。** (為了只看 TF 部分的改變，下面的例子使用和 1-3 **完全相同的 IDF 公式**：$\log\frac{文件總數}{含有這個詞的文件數}$。)


TF-IDF 的 TF 部分就是次數 $f$。BM25 把它換成：

$$\text{TF 部分} = \frac{f \times (k_1 + 1)}{f + k_1 \times L}
\qquad\qquad
L = 1 - b + b \times \frac{\text{這個 chunk 的長度}}{\text{chunk 的平均長度}}$$

| 符號 | 意思 | 常用值 |
|---|---|---|
| $f$ | 這個詞在這個 chunk 中出現幾次 | |
| $k_1$ | 控制「飽和」的速度 | 1.5 |
| $b$ | 控制「長度修正」的強度：0 = 不修正，1 = 完全修正 | 0.75 |
| $L$ | 長度修正：長度等於平均時 $L = 1$；比平均長時 $L > 1$；比平均短時 $L < 1$ | |

這個 TF 部分修正了兩個問題：

1. **出現次數會飽和。** $f$ 同時出現在分子和分母，所以 $f$ 越大，每多出現一次，加的分越少；不管出現幾次，TF 部分都不會超過 $k_1 + 1 = 2.5$。一個詞出現 3 次，不代表「相關 3 倍」。
2. **長的 chunk 會被扣分。** $L$ 乘在分母的 $k_1$ 上。chunk 越長，$L$ 越大，分母越大，分數就越小。很長的 chunk 字多，本來就比較容易碰巧包含問題中的詞，所以它要出現更多次，才能拿到同樣的分數。

**一個 chunk 的總分** = 問題中每一個詞的「IDF × TF 部分」加起來。

下面用 6 個已經斷好詞的短 chunk 示範。問題只有一個詞：「停修」。

- A 和 B 都只出現 1 次「停修」，但 B 的長度是 A 的 4 倍。
- C 出現 3 次「停修」，長度剛好等於平均。
- D、E、F 沒有「停修」。

In [115]:
import math

# 6 個已經斷好詞的短 chunk（詞和詞之間用空白隔開）
bm25_docs = {
    "A 短條文":   "停修 科目 以 2 科 為限",
    "B 長條文":   "學生 申請 停修 後 總 學分數 不得 少於 當學期 應修 最低 學分數 延長 修業 年限 學生 至少 仍應 修習 一個 科目 並 依 規定",
    "C 重複出現": "停修 申請 經 系主任 同意 停修 後 不得 取消 停修 學分費 不退",
    "D 圖書館":   "圖書館 借書 期限 三十 日 逾期 每日 罰款 五 元 上限 五百",
    "E 請假":     "請假 應 檢附 就醫 證明 或 家長 證明 文件",
    "F 獎懲":     "考試 舞弊 者 予以 大過 處分 依 規定 辦理",
}
word = "停修"   # 問題中的詞
k1, b = 1.5, 0.75

tokens = {name: text.split() for name, text in bm25_docs.items()}
N = len(tokens)                                          # chunk 總數
avg_len = sum(len(t) for t in tokens.values()) / N      # 平均長度
n = sum(word in t for t in tokens.values())             # 含有這個詞的 chunk 數


# IDF：和 1-3 相同的公式 log(文件總數 / 含有這個詞的文件數)
# 實際的 BM25 程式（例如後面使用的 rank_bm25 套件）會用加了平滑項的 IDF 版本，
# 所以 search_bm25 的分數不會和這張表相同，但想法一樣：含有這個詞的文件越少，IDF 越大
idf = math.log(N / n)

""" # IDF：越稀有越重要。BM25 常用的版本是 log(1 + (N - n + 0.5) / (n + 0.5))
# （rank_bm25 套件的 IDF 公式又稍有不同，所以後面 search_bm25 的分數不會和這張表相同，但想法一樣）
idf = math.log(1 + (N - n + 0.5) / (n + 0.5)) """

print(f"chunk 總數 N = {N}，平均長度 = {avg_len:.0f} 個詞，含有「{word}」的 chunk 數 n = {n}，IDF = {idf:.2f}")

rows = []
for name, t in tokens.items():
    f = t.count(word)                                    # 出現次數
    L = 1 - b + b * len(t) / avg_len                     # 長度修正
    tf_part = f * (k1 + 1) / (f + k1 * L)                # BM25 的 TF 部分

    rows.append({
        # 輸入：這個 chunk 的長度、出現次數，以及由長度算出的 L
        "chunk": name,
        "長度": len(t),
        "出現次數 f": f,
        "長度修正 L": L,
        # 比較兩種 TF 部分（BM25 只改這裡）
        "TF-IDF 的 TF 部分 (= f)": f,
        "BM25 的 TF 部分": tf_part,
        # 兩者共用同一個 IDF
        "IDF": idf,
        # 比較兩種最後的分數
        "TF-IDF 分數 (= f × IDF)": f * idf,
        "BM25 分數 (= TF 部分 × IDF)": tf_part * idf,
    })

pd.DataFrame(rows).set_index("chunk").round(2)

chunk 總數 N = 6，平均長度 = 12 個詞，含有「停修」的 chunk 數 n = 3，IDF = 0.69


,長度,出現次數 f,長度修正 L,TF-IDF 的 TF 部分 (= f),BM25 的 TF 部分,IDF,TF-IDF 分數 (= f × IDF),BM25 分數 (= TF 部分 × IDF)
chunk,,,,,,,,
A 短條文,6,1,0.62,1,1.29,0.69,0.69,0.89
B 長條文,24,1,1.75,1,0.69,0.69,0.69,0.48
C 重複出現,12,3,1.00,3,1.67,0.69,2.08,1.16
D 圖書館,12,0,1.00,0,0.00,0.69,0.00,0.00
E 請假,9,0,0.81,0,0.00,0.69,0.00,0.00
F 獎懲,9,0,0.81,0,0.00,0.69,0.00,0.00


**觀察：比較「TF-IDF 分數」和「BM25 分數」兩欄。**

| 比較 | TF-IDF 分數 | BM25 分數 | 原因 |
|---|---|---|---|
| A 和 B（都出現 1 次，B 的長度是 A 的 4 倍） | 同分：0.69 和 0.69 | A 比較高：0.89 和 0.48 | **修正 2（長度）：** B 比平均長，$L = 1.75$，被扣分；A 比平均短，$L = 0.62$，被加分。 |
| C 和 A（C 出現 3 次，A 出現 1 次） | C 是 A 的 3 倍：2.08 和 0.69 | C 只比 A 高一點：1.16 和 0.89 | **修正 1（飽和）：** 多出現的兩次，加的分越來越少。 |
| D、E、F | 0 | 0 | 沒有「停修」，不論長度都是 0。 |

**連結到我們的資料：** 我們的 chunks 長度差很多，最短 8 字，最長 14,350 字（大約是平均長度的 15 倍）。一個詞在最長的 chunk 中只出現 1 次時，$L$ 大約是 11.6，TF 部分只剩大約 0.14。所以長度修正對我們的資料特別重要。


## 1-5｜關鍵字檢索的共同特性

詞袋、TF-IDF、BM25 都屬於**關鍵字檢索**，它們有一個共同的特性：

> **問題和條文必須有「相同的詞」，才找得到。**

這是優點（精確的詞、條號、數字、專有名詞，找得又快又準），也是缺點（換一種說法就找不到）。第 1 節最後我們會親眼看到這個缺點。

最後，中文有一個英文沒有的問題：**詞和詞之間沒有空白**。上面的例子是先用空白隔開的，真正的校規沒有空白，所以要先「斷詞」。


## 1.6 中文斷詞
中文有一個英文沒有的問題：**詞和詞之間沒有空白**。上面的例子是先用空白隔開的，真正的校規沒有空白，所以要先「斷詞」。


## 任務 1-A（教師示範）：jieba 斷詞與 TF-IDF 檢索

### jieba (Chinese for “to stutter”)

**jieba（結巴）** 是最常用的中文斷詞 Python 套件，免費、開放原始碼。它用兩個方法決定詞的邊界：

1. **查詞典：** jieba 內建一本大約 35 萬個詞的詞典，每個詞附有它在大量文字中出現的次數。jieba 先找出句子中所有可能的切法，再選出「由常見的詞組成」機率最高的那一種。
2. **猜測不認識的字：** 詞典裡沒有的字，jieba 用一個統計模型（HMM，隱藏式馬可夫模型）猜哪幾個字應該合成一個詞。

### `jieba.lcut()`

```python
jieba.lcut("要斷詞的句子")   # 回傳一個 list，每個元素是一個詞
```

- `lcut` 的 **l** 代表 **list**：結果直接是一個 list，可以用 `print` 看、可以用 `for` 逐一取出。
- 另一個函式 `jieba.cut()` 做的事情完全一樣，但回傳的不是 list，而是要再用 `list(...)` 轉換才看得到內容。所以我們一律使用 `lcut`。
- 預設是「精確模式」：每個字只屬於一個詞，詞與詞不重疊，適合用來做檢索。

**第一次執行時**會出現類似 `Building prefix dict from the default dictionary ...` 和 `Prefix dict has been built successfully.` 的訊息（可能是紅色字）。這不是錯誤，是 jieba 在載入詞典；每次重新啟動 kernel 後只會出現一次。

In [116]:
import jieba

sentence = "每學期申請停修科目以2科為限只限於選修課"
print(jieba.lcut(sentence))

['每學期', '申請', '停修', '科目', '以', '2', '科', '為限', '只限', '於', '選修課']


觀察上面的結果，找找看哪裡切錯了。

預期會看到類似這樣的結果：`['每學期', '申請', '停修', '科目', '以', '2', '科為', '限', '只限', '於', '選修課']`

- 「停修」「選修課」切對了。
- 但是「2科為限」被切成 `'2', '科為', '限'`，正確應該是 `'2', '科', '為限'`。

**為什麼會切錯？** jieba 內建的詞典以簡體中文為主，很多繁體字（例如「為」）的詞不在詞典裡。遇到不認識的字，jieba 只能用統計模型「猜」哪幾個字是一個詞，所以有時猜對（停修），有時猜錯（科為）。

**切錯會怎樣？** 關鍵字檢索是用「詞」來配對的。條文被切成「科為」，但使用者問「幾科」時切出來的是「科」，兩邊的詞不一樣，就配對不到。

我們可以用 `add_word` 把重要的詞加進詞典，讓 jieba 不必用猜的。

In [117]:
for w in ["為限", "停修", "延長修業年限", "選修課", "必修課", "借期", "病假"]:
    jieba.add_word(w)

print(jieba.lcut(sentence))

['每學期', '申請', '停修', '科目', '以', '2', '科', '為限', '只限', '於', '選修課']


In [118]:
# 斷詞函式：切詞後，丟掉標點符號、空白、Markdown 的 # 號
def tokenize(text):
    return [t for t in jieba.lcut(text) if any(ch.isalnum() for ch in t)]

print(tokenize(df.iloc[2]["text"]))

['第二', '條', '適用', '範圍', '每學期', '申請', '停修', '科目', '以', '2', '科', '為限', '只限', '於', '選修課', '惟', '停修', '後', '之', '總', '學分數', '不得', '少', '於', '當學期', '應修', '最低', '學分數', '延長修業年限', '學生', '停修', '後', '至少', '仍應', '修習', '一個', '科目']


In [119]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 建立 TF-IDF 轉換器，三個參數的意思：
tfidf = TfidfVectorizer(
    tokenizer=tokenize,   # 用我們自己寫的 tokenize（jieba 斷詞）把文字切成詞。
                          #   預設的切法是「用空白和標點分開」，中文沒有空白，
                          #   一整串中文會被當成一個詞，所以一定要換成 jieba。
    token_pattern=None,   # 預設的切詞規則（一個正則表達式）。既然已經用 tokenizer，
                          #   就用不到它；設成 None，scikit-learn 才不會顯示警告。
    lowercase=False,      # 不把英文轉成小寫。中文沒有大小寫，不受影響；
                          #   英文的 Students 和 students 會被當成兩個不同的詞。
)

# fit_transform 一次做兩件事：
#   fit：讀過全部 chunks，建立詞彙表（每個詞一欄），並計算每個詞的 IDF
#   transform：把每個 chunk 變成一列 TF-IDF 向量
# 另外，scikit-learn 預設會把每一列的向量長度調成 1，所以兩列做內積，就等於 cosine 相似度
X_tfidf = tfidf.fit_transform(texts)

# X_tfidf 的大小是 (chunks 數, 詞彙數)：每個 chunk 是一列，每個詞是一欄
print("矩陣大小 (chunks 數, 詞彙數)：", X_tfidf.shape)

# nnz = number of non-zero，不是 0 的格子數；除以總格子數，就是不是 0 的比例
print(f"其中不是 0 的格子只佔 {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.2%}  → 這種向量叫做「稀疏向量」")

矩陣大小 (chunks 數, 詞彙數)： (345, 8203)
其中不是 0 的格子只佔 1.02%  → 這種向量叫做「稀疏向量」


In [120]:
# 第 2 個 chunk（停修要點第二條）中，TF-IDF 權重最高的 5 個詞
vocab = np.array(tfidf.get_feature_names_out())
row = X_tfidf[2].toarray().ravel()
for j in np.argsort(-row)[:5]:
    print(f"{vocab[j]}\t{row[j]:.3f}")

停修	0.459
學分數	0.278
科目	0.263
延長修業年限	0.207
只限	0.193


In [121]:
# 任務 1-A（教師示範）：TF-IDF 檢索
def search_tfidf(query, k=3):
    # 用 TF-IDF 檢索：回傳和問題最相似的 k 個 chunks，格式是 [(列號, 分數), ...]

    # 1. 問題也變成 TF-IDF 向量
    #    用 transform，不用 fit_transform：沿用 chunks 建立的詞彙表和 IDF，
    #    這樣問題的每一欄才會和 chunks 的每一欄對應到同一個詞。
    #    問題中出現、但詞彙表裡沒有的詞，會被忽略。
    #    [query] 要放在 list 裡，因為 transform 一次可以處理很多段文字。
    q = tfidf.transform([query])                    # 大小：(1, 詞彙數)

    # 2. 問題和每個 chunk 算相似度
    #    X_tfidf @ q.T 是矩陣乘法：(chunks 數, 詞彙數) × (詞彙數, 1) = (chunks 數, 1)
    #    每一列都是「一個 chunk 和問題的內積」；因為向量長度都是 1，內積就是 cosine 相似度。
    #    .toarray() 把稀疏矩陣轉成一般的 numpy 陣列，.ravel() 把它攤平成一排數字。
    scores = (X_tfidf @ q.T).toarray().ravel()      # 大小：(chunks 數,)

    # 3. 分數由高到低，取前 k 個
    #    np.argsort 回傳「由小到大排序後的位置」；加上負號 -scores，就變成由大到小。
    order = np.argsort(-scores)[:k]

    # 4. 回傳 (列號, 分數)；int() 和 float() 把 numpy 的數字轉成一般的 Python 數字
    return [(int(i), float(scores[i])) for i in order]


question = "每學期申請停修科目以幾科為限？"
print("TFIDF")
show(question, search_tfidf(question))

TFIDF
問題：每學期申請停修科目以幾科為限？
  1. [0.515] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [0.305] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…
  3. [0.293] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…


## 任務 1-B（兩人一組）：BM25 檢索

`rank_bm25` 套件的用法：
- `BM25Okapi(斷好詞的 chunks)`：建立 BM25 模型
- `bm25.get_scores(斷好詞的問題)`：回傳問題對每一個 chunk 的分數（numpy 陣列）

請模仿上面的 `search_tfidf`，完成 `search_bm25` 中的兩行 TODO。

In [122]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized_chunks)

def search_bm25(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]

In [123]:
# 檢查 1-B
_res = search_bm25("每學期申請停修科目以幾科為限？")
assert len(_res) == 3, "應該回傳 3 筆結果"
assert _res[0][1] >= _res[1][1] >= _res[2][1], "分數應該由高到低排列"
print("1-B 完成")
print("BM25: ")
show("每學期申請停修科目以幾科為限？", _res)

1-B 完成
BM25: 
問題：每學期申請停修科目以幾科為限？
  1. [25.666] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [14.373] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…
  3. [10.988] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第三條 受理時間  每學期辦理停修期間以學期第10-11週為原則，四技三年級上  學期以學期第8-9週為原則，詳細受理日期應明訂於學校…


**討論：** 把問題改成口語的問法：「這學期課太重，想中途放棄一門課，最多可以放棄幾門？」意思和上一題相同，答案也是停修要點的**第二條**。

執行下一格，然後討論：

1. TF-IDF 和 BM25 的前 3 名中，有沒有第二條？
2. 每個結果和問題「相同的詞」是哪些？（下一格會列出來）這些詞和「停修」的意思有關嗎？
3. 和上一題比較，這一題的分數高還是低？

In [124]:
# 列出每個結果中，和問題「相同的詞」各出現幾次
def matched_words(query, top):
    q_words = list(dict.fromkeys(tokenize(query)))      # 問題中的詞（去掉重複）
    for rank, (i, score) in enumerate(top, 1):
        chunk_words = tokenize(df.loc[i, "text"])
        same = {w: chunk_words.count(w) for w in q_words if w in chunk_words}
        print(f"  {rank}. {df.loc[i, 'heading'][:20]}　相同的詞：{same}")

q = "這學期課太重，想中途放棄一門課，最多可以放棄幾門？"
print("問題斷詞：", tokenize(q), "\n")

print("━━ TF-IDF ━━")
top = search_tfidf(q)
show(q, top)
matched_words(q, top)

print("\n━━ BM25 ━━")
top = search_bm25(q)
show(q, top)
matched_words(q, top)

問題斷詞： ['這學期', '課', '太重', '想', '中途', '放棄', '一門', '課', '最', '多', '可以', '放棄', '幾門'] 

━━ TF-IDF ━━
問題：這學期課太重，想中途放棄一門課，最多可以放棄幾門？
  1. [0.085] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第一條 目的  為顧及學生於加退選課程截止後因特殊情形無法繼續修習課 程，特訂定本校學生期中停修課程實施要點(以下簡稱本要 點)。…
  2. [0.080] Student_Affairs｜A242060013_學生請假作業辦法.pdf
       ## 十、 心理健康假：  因心理不適或精神狀況不佳，致上課有困難時，得請心 理健康假(每次最多 2 天，每學期最多 7 天)。每學期第 1…
  3. [0.053] Academic_Affairs｜A232060013_學生抵免學分辦法.pdf
       ## 第二條 適用對象  - 一、 轉系（組）生、新生、轉學生。 - 二、 法令准許先修讀學分後考取修讀學位者。 - 三、 申請放棄選讀輔系…
  1. ## 第一條 目的　相同的詞：{'課': 1}
  2. ## 十、 心理健康假：　相同的詞：{'最': 1, '多': 1}
  3. ## 第二條 適用對象　相同的詞：{'放棄': 1}

━━ BM25 ━━
問題：這學期課太重，想中途放棄一門課，最多可以放棄幾門？
  1. [12.652] Student_Affairs｜A242060013_學生請假作業辦法.pdf
       ## 十、 心理健康假：  因心理不適或精神狀況不佳，致上課有困難時，得請心 理健康假(每次最多 2 天，每學期最多 7 天)。每學期第 1…
  2. [10.767] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第一條 目的  為顧及學生於加退選課程截止後因特殊情形無法繼續修習課 程，特訂定本校學生期中停修課程實施要點(以下簡稱本要 點)。…
  3. [10.236] Academic_Affairs｜A232060013_學生抵免學分辦法.pdf
     

## 第 1 節小結

- 電腦比較文字的方法：先把文字變成**向量**，再比較向量。
- 關鍵字檢索（TF-IDF、BM25）的向量是「數詞」得來的：一個詞一個欄位，大部分是 0。
- 中文要先**斷詞**；斷詞切錯，檢索就會受影響。
- 關鍵字檢索的限制：**問題和條文沒有相同的詞，就找不到**，即使意思完全一樣。


---
# 第 2 節｜語意檢索 (10:00–10:50)

## 講解 2-1｜關鍵字檢索的限制：

第 1 節最後，「放棄一門課」找不到「停修」的條文。兩句話意思相同，但**沒有任何一個詞相同**，所以關鍵字檢索的分數是 0。

在詞袋和 TF-IDF 中，每個詞是一個獨立的欄位。對電腦來說，「停修」和「放棄課程」的關係，跟「停修」和「圖書館」的關係**完全一樣：都是不同的欄位，毫無關聯**。

我們需要另一種向量：**意思相近的文字，向量也相近。**



## 2-2｜Word2Vec (2013)：從上下文學到詞的意思

**核心想法：一個詞的意思，可以從它周圍常出現的詞看出來。**

看下面兩個句子，即使你不認識「停修」這個詞，也猜得到它和「退選」意思接近，因為它們出現在幾乎相同的上下文中：

- 學生向教務處申請 **停修** 這門課
- 學生向教務處申請 **退選** 這門課

Word2Vec 讓電腦讀大量的文字，反覆做一個練習：「看到一個詞，猜它旁邊會出現什麼詞」。練習完之後，每個詞會得到一串數字（通常是幾百個），這串數字叫做**詞向量 (word embedding)**。上下文相似的詞，詞向量也會相似。

**和 TF-IDF 的差別：**

| | TF-IDF 向量 | 詞向量 / 句向量 |
|---|---|---|
| 長度 | 幾千到幾萬（一個詞一欄） | 幾百 |
| 內容 | 大部分是 0 → **稀疏 (sparse)** | 每一格都有數值 → **稠密 (dense)** |
| 每一格的意思 | 對應一個特定的詞 | 單獨一格沒有可以解釋的意思 |
| 如何產生 | 根據詞的出現次數 | 模型從大量文字中學來 |
| 意思相近的詞 | 沒有關聯 | 向量相近 |

**Word2Vec 的兩個限制：**

1. **一個詞只有一個向量。** 「蘋果」是水果還是公司？英文的 bank 是銀行還是河岸？ Word2Vec 分不出來。
2. **只有詞的向量，沒有句子的向量。** 最簡單的做法是把句子中所有詞的向量平均，但這樣會**失去詞的順序**。下面這一格示範：只看有哪些字、不看順序的話，兩句意思相反的話會變成一模一樣的向量。

In [125]:
cv_char = CountVectorizer(analyzer="char")
order_demo = ["老師評分學生", "學生評分老師"]
pd.DataFrame(cv_char.fit_transform(order_demo).toarray(), index=order_demo, columns=cv_char.get_feature_names_out())

,分,學,師,生,老,評
老師評分學生,1,1,1,1,1,1
學生評分老師,1,1,1,1,1,1


## 2-3｜RNN / LSTM：依照順序讀句子

為了保留順序，下一代的模型**一個字一個字依序讀**，每讀一個字，就更新一份「到目前為止讀到了什麼」的筆記（叫做**隱藏狀態 hidden state**）。讀完最後一個字，這份筆記就代表整句話。

```
 老師 → 評分 → 學生            學生 → 評分 → 老師
  │      │      │              │      │      │
  ▼      ▼      ▼              ▼      ▼      ▼
[筆記1]→[筆記2]→[筆記3]        [筆記1]→[筆記2]→[筆記3]
                 ↑                              ↑
          代表「老師評分學生」            代表「學生評分老師」
```

這種模型叫做 **RNN**（循環神經網路）。**LSTM** 是 RNN 的改良版（1997 年提出，大約 2014–2017 年間是自然語言處理的主流），它加上了「決定要記住什麼、忘記什麼」的機制，讓筆記可以保存得比較久。

**兩個限制：**

1. **句子一長，前面的內容還是會被沖淡。** 一條 500 字的校規，讀到最後時，開頭的內容已經所剩無幾。
2. **必須依序計算，無法平行處理，速度慢。** 第 100 個字要等前面 99 個字都算完。


- 意思無關的一對 → 把兩個向量**推遠**


## 2-4｜Attention 與 Transformer (2017)

**Attention（注意力機制）的想法：** 不要只靠一份一路傳下去的筆記。處理每一個字的時候，讓它**直接去看句子中所有其他的字**，並決定哪些字和自己最有關係。

- 距離不再是問題：第 1 個字和第 500 個字可以直接互相參考。
- 所有的字可以同時計算，速度快很多。

完全以 Attention 為基礎的模型架構叫做 **Transformer**。BERT、GPT，以及今天使用的句向量模型，都是 Transformer。（第 12 週會詳細介紹。）

## 2-5｜句向量模型：今天要用的工具

**句向量模型 (sentence embedding model)** 用 Transformer 讀完整段文字，輸出**一個向量**代表整段文字的意思。

它如何學會「意思相近 → 向量相近」？訓練時給它大量成對的句子：

- 意思相近的一對（例如一個問題和它的正確答案）→ 把兩個向量**拉近**
反覆幾億次之後，模型就學會把意思相近的文字放在向量空間中相近的位置。

**技術演進整理：**

| 年代 | 方法 | 解決了什麼 | 還有什麼限制 |
|---|---|---|---|
| — | 詞袋 BoW / TF-IDF / BM25 | 把文字變成數字 | 不同的詞之間毫無關聯 |
| 2013 | Word2Vec | 意思相近的詞，向量相近 | 一詞一向量；沒有句子向量 |
| 2014–2017 | RNN / LSTM | 依序讀，保留詞的順序 | 長句會遺忘；計算慢 |
| 2017 | Transformer | 每個字直接看所有的字；可平行計算 | 需要大量資料與運算來訓練 |
| 2019– | 句向量模型（以 Transformer 為基礎, Sentence-BERT、BGE…） | 整段文字 → 一個向量 | 一次能讀的長度有上限 |

## 任務 2-A：cosine 相似度與句向量

有了向量，還需要一個方法來量「兩個向量有多相近」。最常用的是 **cosine 相似度**：看兩個向量的**夾角**，不看長度。

- 方向完全相同 → 1
- 互相垂直（毫無關係）→ 0

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

請完成下面的 `cosine` 函式（一行）。提示：內積用 `np.dot(a, b)`，向量長度用 `np.linalg.norm(a)`。

In [126]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [127]:
# 檢查 TODO 3
assert abs(cosine(np.array([1.0, 0.0]), np.array([2.0, 0.0])) - 1) < 1e-6, "方向相同應該是 1"
assert abs(cosine(np.array([1.0, 0.0]), np.array([0.0, 3.0]))) < 1e-6, "互相垂直應該是 0"
print("cosine 完成")

cosine 完成


**先用手動設定的小例子感受一下。** 下面四個詞各用 2 個數字表示。這些數字是**為了示範而手動設定的**（真正的詞向量有幾百個數字，而且是模型學出來的）。第一個數字代表「和修課有多相關」，第二個數字代表「和圖書館有多相關」。

In [128]:
word_vec = pd.DataFrame(
    {"和修課有關": [0.9, 0.8, 0.1, 0.2], "和圖書館有關": [0.1, 0.2, 0.9, 0.8]},
    index=["停修", "放棄課程", "借書", "還書"],
)
display(word_vec)

# 用你寫的 cosine 函式，算出每兩個詞之間的相似度
pd.DataFrame(
    [[round(cosine(word_vec.loc[a].values, word_vec.loc[b].values), 2) for b in word_vec.index] for a in word_vec.index],
    index=word_vec.index, columns=word_vec.index,
)

,和修課有關,和圖書館有關
停修,0.9,0.1
放棄課程,0.8,0.2
借書,0.1,0.9
還書,0.2,0.8


,停修,放棄課程,借書,還書
停修,1.00,0.99,0.22,0.35
放棄課程,0.99,1.00,0.35,0.47
借書,0.22,0.35,1.00,0.99
還書,0.35,0.47,0.99,1.00


**觀察：** 「停修」和「放棄課程」的相似度接近 1，雖然它們沒有任何相同的字；「停修」和「借書」的相似度很低。這就是我們想要的效果。接下來換成真正的模型。

載入句向量模型。我們使用 `BAAI/bge-small-zh-v1.5`：體積小、支援中文、一般筆電的 CPU 就能執行。

模型已經在課堂一開始從數位學習平台下載，放在專案的 `models/` 資料夾中，所以這一格只需要載入（第一次 `import` 大約需要十幾秒）。如果這一格開始下載或出現錯誤，請回到 `10_0_setup.ipynb` 確認步驟 2 和步驟 6。

In [129]:
import time
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-zh-v1.5"
LOCAL_MODEL = ROOT / "models" / "bge-small-zh-v1.5"   # 從數位學習平台下載的模型
model = SentenceTransformer(str(LOCAL_MODEL) if (LOCAL_MODEL / "config.json").exists() else MODEL_NAME)

toy = ["我想停修一門課", "我想中途放棄一個科目", "圖書館幾點關門"]
toy_emb = model.encode(toy)
print("每個句子變成一個向量，大小：", toy_emb.shape)
print("第一句的前 8 個數字：", np.round(toy_emb[0][:8], 3))

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

每個句子變成一個向量，大小： (3, 512)
第一句的前 8 個數字： [-0.013  0.084  0.031  0.003  0.017 -0.057 -0.036  0.058]


In [130]:
# 同樣三句話，比較「TF-IDF 的相似度」和「句向量的相似度」
toy_tfidf = tfidf.transform(toy).toarray()   # TF-IDF 向量長度已經是 1，內積就是 cosine

for a, b in [(0, 1), (0, 2)]:
    print(f"「{toy[a]}」 vs 「{toy[b]}」")
    print(f"   TF-IDF 相似度：{float(toy_tfidf[a] @ toy_tfidf[b]):.3f}")
    print(f"   句向量相似度：{cosine(toy_emb[a], toy_emb[b]):.3f}")

「我想停修一門課」 vs 「我想中途放棄一個科目」
   TF-IDF 相似度：0.000
   句向量相似度：0.660
「我想停修一門課」 vs 「圖書館幾點關門」
   TF-IDF 相似度：0.000
   句向量相似度：0.537


**觀察：** 第 1 句和第 2 句沒有共同的關鍵詞，TF-IDF 認為它們不太像；句向量則認為它們比「圖書館幾點關門」相近得多。這就是「語意」檢索。

再看一次「狗咬人」和「人咬狗」。只數字的時候，這兩句的向量一模一樣。句向量模型會考慮順序，所以兩個向量不會完全相同（相似度小於 1）。

In [131]:
order_emb = model.encode(order_demo)
print(f"「{order_demo[0]}」 vs 「{order_demo[1]}」 句向量相似度：{cosine(order_emb[0], order_emb[1]):.3f}")

「老師評分學生」 vs 「學生評分老師」 句向量相似度：0.983


## 任務 2-B：用句向量檢索 chunks

步驟和關鍵字檢索完全一樣，只是換了一種向量：
1. 把所有 chunks 變成句向量（已經寫好）
2. 把問題變成句向量
3. 算問題和每個 chunk 的 cosine 相似度，取前 k 個

`normalize_embeddings=True` 會把每個向量的長度調成 1。長度都是 1 的時候，cosine 相似度就等於內積，所以可以用 `chunk_emb @ q` 一次算完全部 chunks。

注意：這個模型一次最多讀大約 512 個 token，太長的 chunk 後半段會被截掉。這是切塊大小會影響檢索的原因之一。

In [132]:
# 把所有 chunks 變成句向量（在 CPU 上可能需要幾十秒到幾分鐘）
# 10_0_setup.ipynb 已經用完全相同的程式預先算好並存成檔案，所以這裡通常會直接讀檔；
# 如果檔案不存在，就在這裡計算
EMB_PATH = ROOT / "data" / "embeddings" / "week10_chunk_emb.npy"

if EMB_PATH.exists() and np.load(EMB_PATH).shape[0] == len(texts):
    chunk_emb = np.load(EMB_PATH)
    print("已從檔案讀取：", EMB_PATH)
else:
    t0 = time.time()
    chunk_emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=16)
    EMB_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(EMB_PATH, chunk_emb)
    print(f"計算時間：{time.time() - t0:.1f} 秒，已儲存：{EMB_PATH}")

print("chunk 向量矩陣大小 (chunks 數, 向量維度)：", chunk_emb.shape)

已從檔案讀取： c:\Users\user\Desktop\weiwei\U12627006-mcut\data\embeddings\week10_chunk_emb.npy
chunk 向量矩陣大小 (chunks 數, 向量維度)： (345, 512)


In [133]:
def search_dense(query, k=3):
    q = model.encode([query], normalize_embeddings=True)[0]   # TODO 4
    scores = chunk_emb @ q                                    # TODO 5
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]

In [134]:
# 檢查 2-B
_res = search_dense("這學期課太重，想中途放棄一門課，最多可以放棄幾門？")
assert len(_res) == 3, "應該回傳 3 筆結果"
assert _res[0][1] >= _res[1][1] >= _res[2][1], "分數應該由高到低排列"
print("2-B 完成")
show("這學期課太重，想中途放棄一門課，最多可以放棄幾門？", _res)

2-B 完成
問題：這學期課太重，想中途放棄一門課，最多可以放棄幾門？
  1. [0.676] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [0.668] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第一條 目的  為顧及學生於加退選課程截止後因特殊情形無法繼續修習課 程，特訂定本校學生期中停修課程實施要點(以下簡稱本要 點)。…
  3. [0.666] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…


## 第 2 節小結

- 句向量模型把**一整段文字**變成**一個稠密向量**；意思相近的文字，向量相近。
- 用 **cosine 相似度**測量兩個向量有多相近。
- 語意檢索的步驟和關鍵字檢索相同：chunks 變向量 → 問題變向量 → 算相似度 → 取前 k 個。差別只在「向量是如何得來的」。
- 限制：模型一次能讀的長度有上限，太長的 chunk 後半段會被忽略。

---
# 第 3 節 Session 3｜兩種檢索的比較




## 3-1｜如何比較哪一種檢索比較好？

只檢查一兩個例子無法下結論。比較檢索方法的標準做法是：

1. 準備一組**測試問題**，而且事先知道每一題的正確條文是哪一條。
2. 每一題都用兩種方法各檢索一次，取前 `k` 名 (e.g., 前 3 名)。
3. 檢查前 `k` 名中**有沒有正確的條文**。有，就算「命中 (hit)」。
4. 計算命中題的比例，叫做 **`hit@k`**。例如 10 題中有 7 題命中，`hit@3 = 0.7`。

今天我們用 10 個問題、`k = 3`，並且用**人工判斷**有沒有命中。下週會改成讓程式自動計算。

## 任務 3-A：同一個意思，兩種問法

下面是 10 個校規問題。單數題使用**和條文相同的用詞**；雙數題是**口語、換句話說**的問法。

In [135]:
QUESTIONS = {
    # --- 教務 Academic_Affairs ---
    "Q01": "每學期申請停修科目以幾科為限？",
    "Q02": "這學期課太重，想中途放棄一門課，最多可以放棄幾門？",
    # --- 圖書資訊 Library_Information ---
    "Q03": "大學部學生借書期限及冊數為何？",
    "Q04": "我跟圖書館借的書可以放在我這邊多久？",
    "Q05": "借書逾期罰款每日多少元？",
    "Q06": "書太晚還會被罰錢嗎？",
    # --- 學務 Student_Affairs ---
    "Q07": "請病假應檢附什麼證明？",
    "Q08": "我感冒發燒沒去上課，事後要補交什麼文件？",
    "Q09": "考試舞弊者予以何種處分？",
    "Q10": "考試偷看別人答案被抓到會怎樣？",
}

METHODS = {"tfidf": search_tfidf, "bm25": search_bm25, "dense": search_dense}

def compare(query, k=3):
    for name, fn in METHODS.items():
        print(f"━━ {name} ━━")
        show(query, fn(query, k), k)
    print()

In [137]:
compare(QUESTIONS["Q01"])
compare(QUESTIONS["Q02"])

━━ tfidf ━━
問題：每學期申請停修科目以幾科為限？
  1. [0.515] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [0.305] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…
  3. [0.293] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…
━━ bm25 ━━
問題：每學期申請停修科目以幾科為限？
  1. [25.666] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [14.373] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…
  3. [10.988] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第三條 受理時間  每學期辦理停修期間以學期第10-11週為原則，四技三年級上  學期以學期第8-9週為原則，詳細受理日期應明訂於學校…
━━ dense ━━
問題：每學期申請停修科目以幾科為限？
  1. [0.882] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第

## 任務 3-B：人工判斷10 個問題的檢索結果

執行下一格，印出全部 10 個問題在三種方法下的前 3 名。

然後**一題一題用人工判斷**：這個方法的前 3 名中，有沒有**真的能回答這個問題的條文**？
- 有 → 填 `1`
- 沒有 → 填 `0`

填在再下一格的 `judgement` 中，順序是 `(tfidf, bm25, dense)`。建議兩人分工：一人看 Q01–Q05，一人看 Q06–Q10，再互相核對。

In [138]:
for qid, q in QUESTIONS.items():
    print(f"================ {qid} ================")
    compare(q)

================ Q01 ================
━━ tfidf ━━
問題：每學期申請停修科目以幾科為限？
  1. [0.515] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [0.305] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…
  3. [0.293] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…
━━ bm25 ━━
問題：每學期申請停修科目以幾科為限？
  1. [25.666] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第二條 適用範圍  每學期申請停修科目以2科為限只限於選修課，惟停修後之總 學分數不得少於當學期應修最低學分數；延長修業年限學生 停修…
  2. [14.373] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第七條 學分費用  停修科目依規定應繳學分費（學分學雜費、電腦實習費、語 言實習費），停修後，已繳者不予退費，未繳者須先完成補 繳後方…
  3. [10.988] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第三條 受理時間  每學期辦理停修期間以學期第10-11週為原則，四技三年級上  學期以學期第8-9週為原則，詳細受理日期應明訂於學校…
━━ dense ━━
問題：每學期申請停修科目以幾科為限？
  1. [0.882] Academic_Affairs｜A23

In [139]:
judgement = {
    "Q01": (1, 1, 1),
    "Q02": (0, 0, 1),
    "Q03": (0, 1, 1),
    "Q04": (0, 0, 0),
    "Q05": (1, 1, 1),
    "Q06": (0, 0, 1),
    "Q07": (1, 1, 1),
    "Q08": (0, 0, 0),
    "Q09": (1, 1, 1),
    "Q10": (0, 0, 1),
}

In [140]:
J = pd.DataFrame.from_dict(judgement, orient="index", columns=list(METHODS))
assert J.notna().all().all(), "還有問題沒有判斷，請把所有 None 改成 1 或 0"
J = J.astype(int)

hit_rate = J.mean().round(2)
print("前 3 名中有正確條文的比例 (hit@3)：")
print(hit_rate.to_string())
print()
print("用詞相同的問題 (單數題)：")
print(J.iloc[0::2].mean().round(2).to_string())
print()
print("換句話說的問題 (雙數題)：")
print(J.iloc[1::2].mean().round(2).to_string())
J

前 3 名中有正確條文的比例 (hit@3)：
tfidf    0.4
bm25     0.5
dense    0.8

用詞相同的問題 (單數題)：
tfidf    0.8
bm25     1.0
dense    1.0

換句話說的問題 (雙數題)：
tfidf    0.0
bm25     0.0
dense    0.6


,tfidf,bm25,dense
Q01,1,1,1
Q02,0,0,1
Q03,0,1,1
Q04,0,0,0
Q05,1,1,1
Q06,0,0,1
Q07,1,1,1
Q08,0,0,0
Q09,1,1,1
Q10,0,0,1


「前 3 名中有沒有正確條文」這個指標就是 `**hit@3**`。

## 3-2｜兩種檢索各自擅長什麼？

| | 關鍵字檢索 (TF-IDF / BM25) | 語意檢索 (句向量) |
|---|---|---|
| 擅長 | 精確的用詞：條號、表單編號、專有名詞、數字 | 口語、換句話說、同義詞 |
| 不擅長 | 問題和條文用詞不同 | 很少見的專有名詞或編號；很長的 chunk |
| 需要的資源 | 很少，速度快 | 需要模型，計算較慢 |
| 結果能不能解釋 | 可以（看得出是哪個詞配對到） | 比較難（只有一個相似度分數） |

因為兩種方法的強項互補，實務上的系統常常**兩種一起用**，再把結果合併，這種做法叫做**混合檢索 (hybrid search)**。

## 任務 3-C：設計你自己的問題

請**自己想 2 個校規問題**（不可以和上面 10 題相同）：
1. 一個你預期**關鍵字檢索比較好**的問題（例如：包含條文編號、專有名詞、精確數字）
2. 一個你預期**語意檢索比較好**的問題（例如：口語、換句話說）

填入下面的 `my_questions`，執行後看結果是否符合預期，並在 `why` 中用一兩句話說明原因。

In [141]:
# TODO 7：填入你的 2 個問題
my_questions = [
    {"question": "表單 A031060104 是用來做什麼的？","expected_better": "keyword","why": "（先執行看結果，再寫，見下方說明）"},
    {"question": "課中途不修了，學費可以退嗎？","expected_better": "dense","why": "（先執行看結果，再寫）"},
]

for item in my_questions:
    assert item["question"].strip(), "請填入問題"
    compare(item["question"])

━━ tfidf ━━
問題：表單 A031060104 是用來做什麼的？
  1. [0.546] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 表單號碼：A031060104…
  2. [0.147] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…
  3. [0.077] Library_Information｜A271010023_圖書館管理作業規範.pdf
       ## 四、個人資料之提供  - (一).您可自由選擇是否提供相關個人資料，惟若拒絕提供個人資料，本館將無法提供相 關服務。 - (二).請依…
━━ bm25 ━━
問題：表單 A031060104 是用來做什麼的？
  1. [17.621] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 表單號碼：A031060104…
  2. [14.511] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 第四條 申請流程  學生申請停修課程應向授課教師提出（表單：  A031060104），經系主任同意後送至教務處課務組辦理。  停修後…
  3. [8.227] Library_Information｜A271010023_圖書館管理作業規範.pdf
       ## 三、個人資料利用之期間、地區、對象與方式  - (一).本館於蒐集目的之存續期間或因執行業務所需保存期間內，得於臺灣地區內合理利 - …
━━ dense ━━
問題：表單 A031060104 是用來做什麼的？
  1. [0.669] Academic_Affairs｜A231060004_學生申請期中停修課程實施要點.pdf
       ## 表單號碼：A031060104…
  2. [0.598] Library_Information｜A271010023_圖書館管理作業規範.pdf
       ## 表號：A

### 反思（請直接在這一格中作答，每題 2–3 句）

1. 在你的結果中，哪一種問題關鍵字檢索比較好？哪一種問題語意檢索比較好？請各舉一題為例。

   > 關鍵字檢索適合精確用詞，語意檢索擅長換句話說，能找到 TF-IDF、BM25 遺漏的相關條文。   

2. 有沒有哪一題是**三種方法都找不到**正確條文的？你認為原因是什麼？（提示：條文是否真的存在於 chunks 中？chunk 是否切得太長或太短？）

   > Q04、Q08 屬檢索失敗，非資料缺漏；長 chunk 可能因 Token 截斷影響檢索。

3. 如果要做校規問答系統，你會選哪一種檢索，或是兩種一起用？為什麼？

   >我會採用混合檢索，結合句向量較高的 hit@3 與關鍵字精確匹配能力，互補優勢以降低正確條文的漏檢率。

## 繳交

執行下一格，產生 `week10_retrieval_comparison.json`。

In [142]:
for item in my_questions:
    assert item["why"].strip(), "請在 my_questions 的 why 中說明原因"

result = {
    "week": 10,
    "model": MODEL_NAME,
    "n_chunks": int(len(df)),
    "questions": QUESTIONS,
    "top3": {
        qid: {name: [df.iloc[i]["chunk_id"] for i, _ in fn(q)] for name, fn in METHODS.items()}
        for qid, q in QUESTIONS.items()
    },
    "judgement": {qid: dict(zip(METHODS, (int(v) for v in vals))) for qid, vals in judgement.items()},
    "hit_at_3": {k: float(v) for k, v in hit_rate.items()},
    "my_questions": my_questions,
}

OUT_PATH = ROOT / "week10_retrieval_comparison.json"
OUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("已儲存：", OUT_PATH)

已儲存： c:\Users\user\Desktop\weiwei\U12627006-mcut\week10_retrieval_comparison.json


### 下課前檢查清單

- [ ] 所有 7 個 `TODO` 檢查都通過，notebook **保留輸出**並存檔
- [ ] 反思 3 題已作答
- [ ] `week10_retrieval_comparison.json` 已產生
- [ ] `requirements.txt` 和 `.gitignore` 已更新（由 `10_0_setup.ipynb` 自動完成）
- [ ] `AI_USAGE.md` 已新增 **Week 10** 段落：今天哪些地方問了 AI、AI 的回答哪裡正確或錯誤、你如何查證
- [ ] `README.md` 已更新：新增 **Week 10** 的說明（本週完成的內容、新增的檔案）
- [ ] commit & push，並**打開 GitHub 網頁確認**檔案已經上傳
